# 08 - Association Rules / Apriori (PuneBite Analyzer)

**Objective:** find "if a restaurant has A, it often also has B" patterns in (a) the cuisines restaurants serve and (b) their amenities and type, using Apriori and FP-Growth.
**Input:** `data/processed/restaurants_clean.csv`
**Output:** `data/processed/association_rules.csv`
**Syllabus link:** Unit VI, association rule mining (Apriori, FP-Growth, support, confidence, lift), Lab 8.

**Mapping to the shopping-basket idea:** one restaurant = one *basket*; its cuisines (or amenities) = the *items* in the basket.

| Term | Meaning | Plain-language reading |
|---|---|---|
| **support**(A) | share of baskets that contain A | how common is A? |
| **confidence**(A -> B) | support(A and B) / support(A) | of restaurants with A, how many also have B? |
| **lift**(A -> B) | confidence / support(B) | how much more often B appears with A than by chance. 1 = no link, above 1 = positive link, below 1 = negative link |

Confidence alone is misleading: if 70% of restaurants have `indoor_seating`, almost any rule ending in it has high confidence. **Lift corrects for that**, so we rank by lift and keep confidence as a filter.

In [ ]:
import os, sys, time, warnings
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/restaurants_clean.csv")
print("Restaurants (baskets):", len(df))

def split_items(s):
    return [x.strip() for x in str(s).split("|") if x.strip()] if isinstance(s, str) else []

# (a) cuisines per restaurant
tx_cuisine = [split_items(s) for s in df["cuisines"]]
# (b) amenities + restaurant type per restaurant (type becomes an item like "TYPE=Casual Dining")
tx_amen = [split_items(a) + ["TYPE=" + str(t)] for a, t in zip(df["amenities"], df["primary_type"])]

for name, tx in [("cuisines", tx_cuisine), ("amenities + type", tx_amen)]:
    sizes = pd.Series([len(t) for t in tx])
    print(f"{name:18s} baskets={len(tx)}  distinct items={len({i for t in tx for i in t})}  items/basket mean={sizes.mean():.2f}, max={sizes.max()}")

## 1. Helper functions
* `one_hot` turns the list of baskets into a True/False table (rows = restaurants, columns = items), which is the input format `mlxtend` needs.
* `mine` runs Apriori **and** FP-Growth, checks that they find the same frequent itemsets (they must: they are two algorithms for the same question) and compares run time.
* `rules_from` builds the rules and keeps those with lift >= 1.2.

In [ ]:
def one_hot(tx):
    te = TransactionEncoder()
    arr = te.fit(tx).transform(tx)
    return pd.DataFrame(arr, columns=te.columns_)

def mine(onehot, min_support):
    t0 = time.perf_counter(); f_ap = apriori(onehot, min_support=min_support, use_colnames=True); t_ap = time.perf_counter() - t0
    t0 = time.perf_counter(); f_fp = fpgrowth(onehot, min_support=min_support, use_colnames=True); t_fp = time.perf_counter() - t0
    same = {(fs, round(s, 10)) for fs, s in zip(f_ap["itemsets"], f_ap["support"])} == \
           {(fs, round(s, 10)) for fs, s in zip(f_fp["itemsets"], f_fp["support"])}
    print(f"min_support={min_support}: Apriori {len(f_ap)} itemsets in {t_ap:.3f}s | FP-Growth {len(f_fp)} itemsets in {t_fp:.3f}s | identical results: {same}")
    return f_ap

def rules_from(freq, n_baskets, min_lift=1.2):
    try:                                   # newer mlxtend versions need the number of baskets
        r = association_rules(freq, num_itemsets=n_baskets, metric="lift", min_threshold=min_lift)
    except TypeError:                      # older versions do not take that argument
        r = association_rules(freq, metric="lift", min_threshold=min_lift)
    return r

def show(r, n=12):
    t = r.copy()
    t["rule"] = t["antecedents"].map(lambda s: ", ".join(sorted(s))) + "  ->  " + t["consequents"].map(lambda s: ", ".join(sorted(s)))
    return t[["rule", "support", "confidence", "lift"]].head(n).round(3).to_string(index=False)

## 2. Basket A: cuisines
Minimum support 0.01 means an itemset must appear in at least about 120 of the 12,134 restaurants; rarer combinations are too thin to trust.

In [ ]:
oh_c = one_hot(tx_cuisine)
print("One-hot table:", oh_c.shape)
freq_c = mine(oh_c, 0.01)
print("Frequent itemsets by size:", freq_c["itemsets"].map(len).value_counts().sort_index().to_dict())
print()
print("Most common single cuisines:")
print(freq_c[freq_c["itemsets"].map(len) == 1].sort_values("support", ascending=False).head(8).assign(itemsets=lambda d: d["itemsets"].map(lambda s: list(s)[0])).round(3).to_string(index=False))

In [ ]:
rules_c = rules_from(freq_c, len(oh_c)).sort_values("lift", ascending=False).reset_index(drop=True)
strong_c = rules_c[rules_c["confidence"] >= 0.5].reset_index(drop=True)
print(f"Rules with lift >= 1.2: {len(rules_c)} | also confidence >= 0.5: {len(strong_c)}")
print()
print("Top cuisine rules by lift (all, confidence not filtered):")
print(show(rules_c))
print()
print("Top cuisine rules with confidence >= 0.5:")
print(show(strong_c))

## 3. Basket B: amenities and restaurant type
There are 86 amenities and many are rare, so the support threshold is 0.02 (about 240 restaurants).

In [ ]:
oh_a = one_hot(tx_amen)
print("One-hot table:", oh_a.shape)
freq_a = mine(oh_a, 0.02)
print("Frequent itemsets by size:", freq_a["itemsets"].map(len).value_counts().sort_index().to_dict())
rules_a = rules_from(freq_a, len(oh_a)).sort_values("lift", ascending=False).reset_index(drop=True)
strong_a = rules_a[rules_a["confidence"] >= 0.5].reset_index(drop=True)
print(f"Rules with lift >= 1.2: {len(rules_a)} | also confidence >= 0.5: {len(strong_a)}")
print()
print("Top amenity/type rules with confidence >= 0.5, by lift:")
print(show(strong_a, 15))

## 4. Which rules are the most useful?
Rules whose consequent is just a very common item (like `indoor_seating`) are true but boring. We look at rules that **end in a restaurant type**, because they answer a question a new owner cares about: "which amenities go with which kind of place?" 

In [ ]:
type_rules = strong_a[strong_a["consequents"].map(lambda s: len(s) == 1 and next(iter(s)).startswith("TYPE="))]
print(f"Rules that predict a restaurant type (confidence >= 0.5): {len(type_rules)}")
print(show(type_rules, 12))

## 5. Visualise: support vs confidence, coloured by lift

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, (name, r) in zip(ax, [("Cuisine rules", rules_c), ("Amenity + type rules", rules_a)]):
    sc = a.scatter(r["support"], r["confidence"], c=r["lift"], cmap="viridis", s=25, alpha=0.8)
    a.set_xlabel("support"); a.set_ylabel("confidence"); a.set_title(name + f" ({len(r)} rules, lift >= 1.2)")
    plt.colorbar(sc, ax=a, label="lift")
plt.tight_layout()

In [ ]:
top = strong_a.head(12).copy()
top["rule"] = top["antecedents"].map(lambda s: ", ".join(sorted(s))) + " -> " + top["consequents"].map(lambda s: ", ".join(sorted(s)))
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top, y="rule", x="lift", color="#4f81bd", ax=ax)
ax.set_title("Top 12 amenity/type rules by lift (confidence >= 0.5)")
plt.tight_layout()

## 6. Save
Sets (frozensets) cannot be saved cleanly in a CSV, so each side of a rule becomes a comma-separated string. The `basket` column says which transaction set a rule came from.

In [ ]:
def tidy(r, basket):
    t = r.copy()
    t["antecedents"] = t["antecedents"].map(lambda s: ", ".join(sorted(s)))
    t["consequents"] = t["consequents"].map(lambda s: ", ".join(sorted(s)))
    t.insert(0, "basket", basket)
    return t[["basket", "antecedents", "consequents", "antecedent support", "consequent support", "support", "confidence", "lift", "leverage", "conviction"]]

all_rules = pd.concat([tidy(rules_c, "cuisines"), tidy(rules_a, "amenities+type")], ignore_index=True)
all_rules["conviction"] = all_rules["conviction"].replace([np.inf], np.nan)
all_rules.round(4).to_csv("../data/processed/association_rules.csv", index=False)
print("Saved", len(all_rules), "rules ->", all_rules["basket"].value_counts().to_dict())

## 7. Summary and findings
* **Cuisines** (min support 0.01): 79 frequent itemsets (27 single cuisines, 38 pairs, 13 triples, 1 quadruple) and 126 rules with lift >= 1.2, of which 41 have confidence >= 0.5. North Indian (47%), Chinese (35%) and Fast Food (25%) dominate, so many rules just reflect that the big three travel together.
* **Strongest cuisine link: Continental and Italian** (lift about 9). Of restaurants serving Italian, 43% also serve Continental, nine times more often than by chance; Italian + North Indian -> Continental has lift 12. Desserts and Bakery (lift 7.8) are the other tight pair.
* **Regional pattern:** Malwani, Seafood and Mughlai restaurants almost always also serve Chinese and North Indian (confidence 60-65%, lift about 2.3-2.5): specialty restaurants hedge with the standard menu.
* **Amenities + type** (min support 0.02): 254 frequent itemsets, 1,318 rules (349 with confidence >= 0.5). The strongest links are partly *definitional* (delivery_only <-> TYPE=Delivery, bakery/dessert parlours <-> desserts_and_bakes) and should not be sold as discoveries.
* **Genuinely useful rules:** `nightlife -> full_bar_available` (confidence 100%, lift 10.7); `smoking_are + table_booking_recommended -> full_bar_available` (87%, lift 9.3); full bar + home delivery + indoor seating -> Casual Dining (75%, lift 3.5). Together they describe the "full-service dining" profile: bar, smoking area, table booking, outdoor seating.
* **Apriori vs FP-Growth:** both return exactly the same itemsets (the notebook checks this). FP-Growth avoids generating candidate sets, so it scales better on large data; on this small dataset both are very fast.
* **Limitations:** rules show co-occurrence, not cause (adding a bar does not make a restaurant Casual Dining). With only 12k baskets and a 0.01-0.02 support floor, rare combinations are invisible. Lift can be inflated for rare items, which is why we kept a support floor.